<a href="https://colab.research.google.com/github/hangyeolKim2003/Cat_code_2/blob/main/Correction_Photon_Loss_On_Cat_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install qutip

In [21]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from qutip import coherent, destroy, wigner, fidelity, expect

# 1. Environment Setup
N = 40
alpha = 2.5
a = destroy(N)
n_op = a.dag() * a
parity_op = (1j * np.pi * n_op).expm()

# 2. Initial State: Even Cat
cat_plus = (coherent(N, alpha) + coherent(N, -alpha)).unit()

# 3. Error Occurrence: Single Photon Loss
error_state = (a * cat_plus).unit()

# 4. Error Detection
syndrome = expect(parity_op, error_state).real

# 5. Recovery Process
if syndrome < 0:
    recovered_state = (a.dag() * error_state).unit()
    message = "Photon loss detected -> Recovery performed"
else:
    recovered_state = error_state
    message = "No error detected"

# 6. Animation Sequence (Refactored for Cleanliness)
fid_val = fidelity(cat_plus, recovered_state)

# Define unique frames and titles
states = [cat_plus, error_state, recovered_state, cat_plus]
titles = [
    "1. Initial State (Even Cat)",
    f"2. Error: Photon Loss (Odd Cat)\nParity: {syndrome:.2f}",
    f"3. Recovered State (Even Cat)\nFidelity: {fid_val:.4f}",
    "4. Reset to Initial (Looping...)"
]

# Create the actual display sequence by repeating the recovery frame (index 2)
# Sequence: [0, 1, 2, 2, 2, 3]
sequence_indices = [0, 1, 2, 2, 2, 3]
plot_states = [states[i] for i in sequence_indices]
plot_titles = [titles[i] for i in sequence_indices]

fig, ax = plt.subplots(figsize=(7, 6))
xvec = np.linspace(-6, 6, 200)

def update(frame):
    ax.clear()
    idx = frame % len(plot_states)
    current_state = plot_states[idx]
    current_title = plot_titles[idx]

    W = wigner(current_state, xvec, xvec)
    ax.contourf(xvec, xvec, W, 100, cmap='RdBu_r')
    ax.set_title(current_title)
    ax.set_xlabel('Re(α)')
    ax.set_ylabel('Im(α)')
    return ax,

# Set interval to 1000ms (1 second)
anim = FuncAnimation(fig, update, frames=len(plot_states), interval=1000, blit=False, repeat=True)
plt.close(fig)

print(f"Result: {message}")
display(HTML(anim.to_html5_video()))

Result: Photon loss detected -> Recovery performed
